In [2]:
import pandas as pd
import plotly.express as px
import numpy as np

# Dataset: Global Energy Mix by Country and Source
df = pd.read_csv('global_energy_mix.csv')

# Source type mapping — reuse from lecture
source_category = {
    'Coal': 'Fossil', 'Oil': 'Fossil', 'Natural Gas': 'Fossil',
    'Nuclear': 'Low-carbon', 'Hydro': 'Low-carbon',
    'Wind': 'Renewable', 'Solar': 'Renewable', 'Other Renewables': 'Renewable'
}
df['Source_Type'] = df['Source'].map(source_category)

print(f"Loaded: {len(df)} rows")
print(df.head(10))

Loaded: 103 rows
         Country         Region            Source  Share_pct     TWh  \
0  United States  North America              Coal         10  1015.0   
1  United States  North America               Oil         35  3220.0   
2  United States  North America       Natural Gas         34  3083.0   
3  United States  North America           Nuclear          9   798.0   
4  United States  North America             Hydro          3   339.0   
5  United States  North America              Wind          4   413.0   
6  United States  North America             Solar          3   325.0   
7  United States  North America  Other Renewables          2   229.0   
8          China           Asia              Coal         58  6929.1   
9          China           Asia               Oil         18  1620.0   

  Source_Type  
0      Fossil  
1      Fossil  
2      Fossil  
3  Low-carbon  
4  Low-carbon  
5   Renewable  
6   Renewable  
7   Renewable  
8      Fossil  
9      Fossil  


**Task 1 — Treemap: fossil fuel dependency by country**





In [4]:

# Task 1 — Fossil fuel treemap

import plotly.express as px

# Filter fossil fuel sources
df_fossil = df.loc[df['Source_Type'] == 'Fossil'].copy()

# Calculate fossil fuel dependency for each country
country_total = df.groupby(['Region', 'Country'])['TWh'].sum().reset_index()
country_fossil = df_fossil.groupby(['Region', 'Country'])['TWh'].sum().reset_index()

dependency = country_fossil.merge(
    country_total,
    on=['Region', 'Country'],
    suffixes=('_fossil', '_total')
)

dependency['Fossil_Share'] = (
    dependency['TWh_fossil'] / dependency['TWh_total'] * 100
)

# Find the most fossil-dependent country
top_country = dependency.loc[dependency['Fossil_Share'].idxmax()]

# CVD-safe colour palette
fossil_colors = {
    'Coal': '#D55E00',
    'Oil': '#0072B2',
    'Natural Gas': '#009E73'
}

# Build treemap
fig = px.treemap(
    df_fossil,
    path=['Region', 'Country', 'Source'],
    values='TWh',
    color='Source',
    color_discrete_map=fossil_colors,
    title=(
        f"Most Fossil-Dependent Country: {top_country['Country']} "
        f"({top_country['Fossil_Share']:.1f}%)"
    )
)

# Show TWh values, not percentages
fig.update_traces(
    texttemplate='%{label}<br>%{value:,.0f} TWh',
    textinfo='label',
    insidetextfont=dict(size=11),
    tiling=dict(pad=3)
)

# Grey out parent nodes (Region and Country)
trace = fig.data[0]
trace.marker.colors = [
    fossil_colors.get(label, '#BDBDBD')
    if label in fossil_colors else '#BDBDBD'
    for label in trace.labels
]

fig.update_layout(
    uniformtext=dict(
        minsize=10,
        mode='hide'
    ),
    margin=dict(t=60, l=10, r=10, b=10)
)

fig.show()


**Task 2 — Treemap vs bar: low-carbon energy by country**

In [5]:

# Task 2 — Low-carbon energy charts

# Filter low-carbon energy
df_low_carbon = df.loc[
    df['Source_Type'] == 'Low-carbon'
].copy()

# Aggregate TWh by country
low_carbon_country = (
    df_low_carbon.groupby('Country', as_index=False)['TWh']
    .sum()
    .sort_values('TWh', ascending=False)
)

# Add dummy root column for treemap
low_carbon_country['All'] = 'Low-carbon'

# Find leading country
top_country = low_carbon_country.iloc[0]

# 1. Treemap
fig1 = px.treemap(
    low_carbon_country,
    path=['All', 'Country'],
    values='TWh',
    color='TWh',
    color_continuous_scale='Blues',
    title='Low-Carbon Energy Distribution by Country'
)

fig1.update_traces(
    texttemplate='%{label}<br>%{value:,.0f} TWh'
)

fig1.update_layout(
    margin=dict(t=60, l=10, r=10, b=10)
)

fig1.show()


# 2. Horizontal bar chart
fig2 = px.bar(
    low_carbon_country,
    x='TWh',
    y='Country',
    orientation='h',
    color='TWh',
    color_continuous_scale='Blues',
    text='TWh',
    title=(
        f"Leading Country in Low-Carbon Energy: "
        f"{top_country['Country']} "
        f"({top_country['TWh']:,.0f} TWh)"
    ),
    labels={
        'TWh': 'Energy Production (TWh)',
        'Country': 'Country'
    }
)

fig2.update_traces(
    texttemplate='%{text:,.0f} TWh',
    textposition='outside'
)

fig2.update_layout(
    yaxis={'categoryorder': 'total ascending'},
    margin=dict(t=70, l=10, r=30, b=10)
)

fig2.show()


**Task 2 — reflection**

**Which chart tells the ranking story more clearly, and why?**


The horizontal bar graph is a better way to show the ranking because the countries have been arranged in an order of low carbon energy generation. It becomes easy to compare data values and determine the highest and lowest countries. The treemap is helpful in showing the total distribution but comparing the country's data becomes difficult.

**Task 3 — 100% stacked bar: which region relies most on coal?**

In [6]:

# Task 3 — 100% Stacked Bar Chart

# Reuse df_fossil from Task 1

# Aggregate TWh by Region and Source
region_source = (
    df_fossil.groupby(['Region', 'Source'])['TWh']
    .sum()
    .reset_index()
)

# Calculate each region's total fossil energy
region_source['Region_Total'] = (
    region_source.groupby('Region')['TWh']
    .transform('sum')
)

# Calculate percentage share
region_source['Share'] = (
    region_source['TWh'] /
    region_source['Region_Total'] * 100
)

# Find coal share for each region
coal_share = region_source[
    region_source['Source'] == 'Coal'
].copy()

coal_share = coal_share.sort_values(
    'Share', ascending=False
)

# Identify the most coal-dependent region
top_region = coal_share.iloc[0]

# Create percentage labels only for Coal
region_source['Coal_Label'] = np.where(
    region_source['Source'] == 'Coal',
    region_source['Share'].map(lambda x: f'{x:.1f}%'),
    ''
)

# Sort regions by coal share
region_order = coal_share['Region'].tolist()

# Colours: Coal same as Task 1, other sources grey
source_colors = {
    'Coal': '#D55E00',
    'Oil': '#A6A6A6',
    'Natural Gas': '#595959'
}

# Build 100% stacked horizontal bar chart
fig = px.bar(
    region_source,
    x='Share',
    y='Region',
    color='Source',
    orientation='h',
    text='Coal_Label',
    barmode='stack',
    category_orders={
        'Source': ['Coal', 'Oil', 'Natural Gas'],
        'Region': region_order
    },
    color_discrete_map=source_colors,
    title=(
        f"Most Coal-Dependent Region: {top_region['Region']} "
        f"({top_region['Share']:.1f}% Coal)"
    ),
    labels={
        'Share': 'Share of Fossil Energy (%)',
        'Region': 'Region',
        'Source': 'Energy Source'
    }
)

fig.update_traces(
    textposition='inside',
    insidetextanchor='middle',
    textfont=dict(color='white')
)

fig.update_layout(
    xaxis=dict(
        range=[0, 100],
        ticksuffix='%',
        title='Share of Fossil Energy (%)'
    ),
    yaxis=dict(
        categoryorder='array',
        categoryarray=region_order,
        autorange='reversed'
    ),
    legend_title='Fossil Source',
    margin=dict(t=70, l=10, r=10, b=10)
)

fig.show()


**Task 3 — reflection**

**Your Task 1 treemap and this stacked bar show the same fossil data. Name one question each chart answers that the other cannot.**  


From the treemap for Task 1, one can see the absolute consumption of fossil fuel (TWh) of each country, as well as the proportion of energy source of each country. From the 100% stacked bar chart, one can see the proportion of each of Coal, Oil, and Natural Gas in each region, which makes it easy to know which region is dependent mostly on coal.